# Data Preparation & Formatting

**Goal:** Load the Bitext customer support dataset, format it for:
1. **SFT (Supervised Fine-Tuning)** — instruction/response pairs in chat template format
2. **DPO (Direct Preference Optimization)** — preference pairs with `prompt`, `chosen`, `rejected`

Both datasets will be saved in HuggingFace `Dataset` format to the `data/` folder.

**Dataset:** [`bitext/Bitext-customer-support-llm-chatbot-training-dataset`](https://huggingface.co/datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset)

## Imports

In [1]:
import os
import random
import pandas as pd
from pathlib import Path
from datasets import load_dataset, Dataset, DatasetDict
from transformers import AutoTokenizer
from huggingface_hub import login

random.seed(42)

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_DIR = Path("../data")
DATA_DIR.mkdir(exist_ok=True)

SFT_SAVE_PATH  = DATA_DIR / "sft_dataset"
DPO_SAVE_PATH  = DATA_DIR / "dpo_dataset"
TEST_SAVE_PATH = DATA_DIR / "test_dataset"

print("Imports done")
print(f"Data will be saved to: {DATA_DIR.resolve()}")

Imports done
Data will be saved to: /Users/anjishnu_mukherjee/Documents/CODING/TMLC_Projects/Cust-Supp-faq-HFModel/data


In [2]:
login(os.environ["HF_TOKEN"])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Loading the Bitext Dataset

In [3]:
print("Loading Bitext customer support dataset...")

raw_dataset = load_dataset(
    "bitext/Bitext-customer-support-llm-chatbot-training-dataset"
)

print(raw_dataset)
print("\nColumn names:", raw_dataset["train"].column_names)
print(f"\nTotal training samples: {len(raw_dataset['train'])}")

Loading Bitext customer support dataset...
DatasetDict({
    train: Dataset({
        features: ['flags', 'instruction', 'category', 'intent', 'response'],
        num_rows: 26872
    })
})

Column names: ['flags', 'instruction', 'category', 'intent', 'response']

Total training samples: 26872


In [4]:
# Preview a few rows
df = raw_dataset["train"].to_pandas()
print("Sample rows:")
display(df.head(3))

print("\nUnique categories:", df["category"].nunique())
print("Unique intents   :", df["intent"].nunique())
print("\nCategory distribution:")
print(df["category"].value_counts())

Sample rows:


,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...



Unique categories: 11
Unique intents   : 27

Category distribution:
category
ACCOUNT         5986
ORDER           3988
REFUND          2992
INVOICE         1999
CONTACT         1999
PAYMENT         1998
FEEDBACK        1997
DELIVERY        1994
SHIPPING        1970
SUBSCRIPTION     999
CANCEL           950
Name: count, dtype: int64


## Format for SFT — Messages Template

We format each row as a structured conversation using the **messages** format, which is compatible with many modern instruction-tuned models.

```python
messages = [
    {
        "role":    "system",
        "content": SYSTEM_PROMPT
    },
    {
        "role":    "user",
        "content": example["instruction"].strip()
    },
    {
        "role":    "assistant",
        "content": example["response"].strip()
    },
]
```

In [5]:
## Loading the tokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)
tokenizer.pad_tokens = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Tokenizer loaded : {MODEL_ID}")
print(f"Tokenizer Chat Template: {tokenizer.chat_template is not None}")

Tokenizer loaded : Qwen/Qwen2.5-0.5B-Instruct
Tokenizer Chat Template: True


## Stratifyied Sampling

- Sampling equally per intent rather than randomly.
- Prevents Large Categories like ACCOUNT from dominating 

### Constants Taken into Consideration

- SFT -> 200 per intent x 27 = 5400 Samples
- DPO -> 100 per intent x 27 = 2700 Samples
- Test -> 50 per intent x 27 = 1350 Samples

In [6]:
SFT_PER_INTENT = 200
DPO_PER_INTENT = 100
TEST_PER_INTENT = 50

def stratified_sampling(dataset, n_per_intent, seed=42):
    df = dataset.to_pandas()

    samples = []

    for _,group in df.groupby("intent"):
        sampled_group = group.sample(
            n=min(n_per_intent, len(group)),
            random_state=seed
        )
        samples.append(sampled_group)

    sampled = pd.concat(samples, ignore_index=True)
    
    sampled = sampled.sample(frac=1, random_state=seed).reset_index(drop=True)
    return Dataset.from_pandas(sampled, preserve_index=False)

In [7]:
sft_raw = stratified_sampling(raw_dataset["train"], SFT_PER_INTENT)
dpo_raw = stratified_sampling(raw_dataset["train"], DPO_PER_INTENT)
test_raw = stratified_sampling(raw_dataset["train"], TEST_PER_INTENT)

print(f"SFT  subset : {len(sft_raw):,} samples")
print(f"DPO  subset : {len(dpo_raw):,} samples")
print(f"Test subset : {len(test_raw):,} samples")
print(f"\nSFT intent distribution (should be ~150 each):")
print(sft_raw.to_pandas()['intent'].value_counts().to_string())

SFT  subset : 5,400 samples
DPO  subset : 2,700 samples
Test subset : 1,350 samples

SFT intent distribution (should be ~150 each):
intent
delivery_options            200
check_payment_methods       200
get_invoice                 200
delete_account              200
create_account              200
complaint                   200
review                      200
check_refund_policy         200
contact_human_agent         200
edit_account                200
cancel_order                200
newsletter_subscription     200
delivery_period             200
check_cancellation_fee      200
place_order                 200
track_order                 200
get_refund                  200
set_up_shipping_address     200
track_refund                200
contact_customer_service    200
switch_account              200
check_invoice               200
change_order                200
payment_issue               200
registration_problems       200
change_shipping_address     200
recover_password            2

In [8]:
SYSTEM_PROMPT ="""You are a knowledgeable and empathetic customer support assistant.
Your role is to help customers resolve their issues quickly and accurately.
Always be polite, clear, and solution-focused."""

def format_sft_sample(example):
    """
    Converts a raw Bitext row into a chat-template formatted string.
    Uses the role/content message structure, then applies the
    tokenizer's chat template to get the final text.
    """
    messages = [
        {"role": "system","content": SYSTEM_PROMPT},
        {"role": "user","content": example["instruction"]},
        {"role": "assistant","content": example["response"]}
    ]

    example["text"] = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return example


# Apply formatting
sft_dataset = sft_raw.map(
    format_sft_sample,
    num_proc=4, # parallelize since formatting is slow
    desc="Formatting SFT"
)

print(f"SFT dataset size: {len(sft_dataset):,} samples")
print("\n── Sample formatted text ──")
print(sft_dataset[0]["text"])

Formatting SFT (num_proc=4):   0%|          | 0/5400 [00:00<?, ? examples/s]

SFT dataset size: 5,400 samples

── Sample formatted text ──
<|im_start|>system
You are a knowledgeable and empathetic customer support assistant.
Your role is to help customers resolve their issues quickly and accurately.
Always be polite, clear, and solution-focused.<|im_end|>
<|im_start|>user
help me to see what delivery options I can choose<|im_end|>
<|im_start|>assistant
How wonderful it is to assist you in uncovering the multitude of delivery options at your disposal! Allow me to guide you through the labyrinth of choices:

1. Standard Shipping: Experience the joy of anticipation as your package arrives within {{Date Range}} business days, paving the way for a delightful unboxing moment.
2. Expedited Shipping: Harness the power of swiftness with this option, allowing your coveted items to reach your doorstep in just {{Date Range}} business days.
3. Overnight Shipping: For those seeking instant gratification, expect your carefully chosen products to be in your hands the very next 

In [9]:
# Train / Validation split 
# 90% train, 10% validation

sft_split = sft_dataset.train_test_split(test_size=0.1, seed=42)

sft_data = DatasetDict({
    "train": sft_split["train"],
    "validation": sft_split["test"]
})

print(sft_data)
print(f"\nTrain samples     : {len(sft_data['train']):,}")
print(f"Validation samples: {len(sft_data['validation']):,}")

DatasetDict({
    train: Dataset({
        features: ['flags', 'instruction', 'category', 'intent', 'response', 'text'],
        num_rows: 4860
    })
    validation: Dataset({
        features: ['flags', 'instruction', 'category', 'intent', 'response', 'text'],
        num_rows: 540
    })
})

Train samples     : 4,860
Validation samples: 540


## Format for DPO — Preference Pairs

DPO requires three fields per sample:
- `prompt`  — the user's question (with system message)
- `chosen`  — the **correct / good** response
- `rejected` — a **wrong / bad** response

### Strategy for generating `rejected` responses
Since Bitext only provides one (correct) response per query, we create `rejected` responses by **mismatching intents**: for each query, we find a response from a **different intent** within the same category. This simulates a real failure case — the model giving a response that is topically related but semantically wrong for the specific query.

In [10]:
def build_dpo_pairs(dataset, seed=42):
    random.seed(seed)
    df = dataset.to_pandas()

    intent_to_responses = df.groupby("intent")["response"].apply(list).to_dict()
    category_to_intents = (
        df.groupby("category")["intent"]
          .apply(lambda x: list(x.unique()))
          .to_dict()
    )

    prompts, chosens, rejecteds = [], [], []
    skipped = 0

    for _, row in df.iterrows():
        other_intents = [
            i for i in category_to_intents.get(row["category"], [])
            if i != row["intent"]
        ]

        if not other_intents:
            skipped += 1
            continue

        rejected_response = random.choice(
            intent_to_responses[random.choice(other_intents)]
        )

        prompt_messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": row["instruction"]},
        ]

        # add_generation_prompt=True appends <|im_start|>assistant\n
        # so chosen/rejected are the bare response strings only
        prompt = tokenizer.apply_chat_template(
            prompt_messages,
            tokenize=False,
            add_generation_prompt=True
        )

        prompts.append(prompt)
        chosens.append(row["response"])
        rejecteds.append(rejected_response)

    print(f"DPO pairs created : {len(prompts):,}")
    print(f"Skipped           : {skipped}")

    return Dataset.from_dict({
        "prompt":   prompts,
        "chosen":   chosens,
        "rejected": rejecteds
    })


dpo_full = build_dpo_pairs(dpo_raw)
print("\nDPO schema:")
print(dpo_full)

DPO pairs created : 2,500
Skipped           : 200

DPO schema:
Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 2500
})


In [11]:
# Inspect a sample
sample = dpo_full[0]
print("═" * 60)
print("PROMPT:")
print(sample["prompt"])
print("\nCHOSEN (correct response):")
print(sample["chosen"])
print("\nREJECTED (wrong-intent response):")
print(sample["rejected"])
print("═" * 60)

════════════════════════════════════════════════════════════
PROMPT:
<|im_start|>system
You are a knowledgeable and empathetic customer support assistant.
Your role is to help customers resolve their issues quickly and accurately.
Always be polite, clear, and solution-focused.<|im_end|>
<|im_start|>user
i dont know how i can see how soon can i expect my order<|im_end|>
<|im_start|>assistant


CHOSEN (correct response):
We understand your uncertainty about how to track the estimated arrival time of your order. To assist you, please provide us with the {{Tracking Number}} or {{Order Number}} associated with your purchase. With that information, we can provide you with an accurate estimate of when you can expect your order to arrive. Your patience is greatly appreciated as we work together to resolve your query.

REJECTED (wrong-intent response):
I'll make it happen! We're thrilled to deliver our products to {{Delivery Country}}, embracing the beauty of the Nordic region. Whether you resi

In [12]:
# Train / Validation split for DPO 
dpo_split = dpo_full.train_test_split(test_size=0.1, seed=42)

dpo_data = DatasetDict({
    "train":      dpo_split["train"],
    "validation": dpo_split["test"]
})

print(dpo_data)
print(f"\nDPO Train samples     : {len(dpo_data['train']):,}")
print(f"DPO Validation samples: {len(dpo_data['validation']):,}")

DatasetDict({
    train: Dataset({
        features: ['prompt', 'chosen', 'rejected'],
        num_rows: 2250
    })
    validation: Dataset({
        features: ['prompt', 'chosen', 'rejected'],
        num_rows: 250
    })
})

DPO Train samples     : 2,250
DPO Validation samples: 250


In [13]:
test_data = test_raw  # no formatting needed — raw columns preserved

print(f"Test subset ready: {len(test_data):,} samples")
print(f"Columns: {test_data.column_names}")

Test subset ready: 1,350 samples
Columns: ['flags', 'instruction', 'category', 'intent', 'response']


## Validate the Datasets

Quick sanity checks before saving.

In [14]:
def validate_sft(ds):
    assert "text" in ds.column_names
    for i, ex in enumerate(ds.select(range(5))):
        assert "system"    in ex["text"], f"Row {i}: missing system"
        assert "user"      in ex["text"], f"Row {i}: missing user"
        assert "assistant" in ex["text"], f"Row {i}: missing assistant"
    print(f"SFT valid — {len(ds):,} samples")

def validate_dpo(ds):
    for col in ["prompt", "chosen", "rejected"]:
        assert col in ds.column_names, f"Missing: {col}"
    for i, ex in enumerate(ds.select(range(5))):
        assert ex["chosen"] != ex["rejected"], f"Row {i}: chosen == rejected"
        assert len(ex["chosen"])   > 0, f"Row {i}: empty chosen"
        assert len(ex["rejected"]) > 0, f"Row {i}: empty rejected"
    print(f"DPO valid — {len(ds):,} samples")

validate_sft(sft_data["train"])
validate_dpo(dpo_data["train"])

SFT valid — 4,860 samples
DPO valid — 2,250 samples


## Save Datasets to Disk

In [15]:
# Saving SFT dataset
print(f"Saving SFT dataset to: {SFT_SAVE_PATH}")
sft_data.save_to_disk(str(SFT_SAVE_PATH))
print(f"SFT dataset saved -> {SFT_SAVE_PATH}")

# Saving DPO dataset
print(f"\nSaving DPO dataset to: {DPO_SAVE_PATH}")
dpo_data.save_to_disk(str(DPO_SAVE_PATH))
print(f"DPO dataset saved -> {DPO_SAVE_PATH}")

# Saving Test dataset
print(f"\nSaving Test dataset to: {TEST_SAVE_PATH}")
test_data.save_to_disk(str(TEST_SAVE_PATH))
print(f"Test dataset saved -> {TEST_SAVE_PATH}")

Saving SFT dataset to: ../data/sft_dataset


Saving the dataset (0/1 shards):   0%|          | 0/4860 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/540 [00:00<?, ? examples/s]

SFT dataset saved -> ../data/sft_dataset

Saving DPO dataset to: ../data/dpo_dataset


Saving the dataset (0/1 shards):   0%|          | 0/2250 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/250 [00:00<?, ? examples/s]

DPO dataset saved -> ../data/dpo_dataset

Saving Test dataset to: ../data/test_dataset


Saving the dataset (0/1 shards):   0%|          | 0/1350 [00:00<?, ? examples/s]

Test dataset saved -> ../data/test_dataset


In [16]:
# Verify saved files can be reloaded
from datasets import load_from_disk

sft_check = load_from_disk(str(SFT_SAVE_PATH))
dpo_check = load_from_disk(str(DPO_SAVE_PATH))
test_check = load_from_disk(str(TEST_SAVE_PATH))

print("Reload check:")
print(f"  SFT  → {sft_check}")
print(f"  DPO  → {dpo_check}")
print(f"  TEST → {test_check}")
print("\nDatasets saved and verified.")

Reload check:
  SFT  → DatasetDict({
    train: Dataset({
        features: ['flags', 'instruction', 'category', 'intent', 'response', 'text'],
        num_rows: 4860
    })
    validation: Dataset({
        features: ['flags', 'instruction', 'category', 'intent', 'response', 'text'],
        num_rows: 540
    })
})
  DPO  → DatasetDict({
    train: Dataset({
        features: ['prompt', 'chosen', 'rejected'],
        num_rows: 2250
    })
    validation: Dataset({
        features: ['prompt', 'chosen', 'rejected'],
        num_rows: 250
    })
})
  TEST → Dataset({
    features: ['flags', 'instruction', 'category', 'intent', 'response'],
    num_rows: 1350
})

Datasets saved and verified.


## Dataset Summary

In [17]:
print("="*55)
print(" DATASET PREPARATION COMPLETE")
print("="*55)
print(f"\nSaved to: {DATA_DIR.resolve()}")
print()
print("SFT Dataset (data/sft_dataset/)")
print(f"  Train      : {len(sft_data['train']):,} samples")
print(f"  Validation : {len(sft_data['validation']):,} samples")
print(f"  Format     : Chat template with system/user/assistant roles")
print()
print("DPO Dataset (data/dpo_dataset/)")
print(f"  Train      : {len(dpo_data['train']):,} samples")
print(f"  Validation : {len(dpo_data['validation']):,} samples")
print(f"  Format     : prompt | chosen | rejected")
print()
print("Test Dataset (data/test_dataset/)")
print(f"  Samples    : {len(test_data):,} samples")
print(f"  Format     : raw columns (instruction/response/category/intent)")

 DATASET PREPARATION COMPLETE

Saved to: /Users/anjishnu_mukherjee/Documents/CODING/TMLC_Projects/Cust-Supp-faq-HFModel/data

SFT Dataset (data/sft_dataset/)
  Train      : 4,860 samples
  Validation : 540 samples
  Format     : Chat template with system/user/assistant roles

DPO Dataset (data/dpo_dataset/)
  Train      : 2,250 samples
  Validation : 250 samples
  Format     : prompt | chosen | rejected

Test Dataset (data/test_dataset/)
  Samples    : 1,350 samples
  Format     : raw columns (instruction/response/category/intent)
